# 1. The independent reference

Every check in these notebooks compares against the **definition** of the flux deficit,

$$P_\gamma(z,p)=\int_{\rho\le p,\ r<1}(1-r^2)^\gamma\,\mathrm dA ,\qquad
r^2=z^2+\rho^2+2z\rho\cos\phi ,$$

integrated numerically in polar coordinates $(\rho,\phi)$ about the occulter centre
(`refcheck.py`). The reference uses none of the paper's derivations and no pizzetti code:
no series, no Green's theorem, no hypergeometric functions. There are two versions:

* `P_fast`: NumPy, double precision, tanh-sinh quadrature in both variables (~10 ms);
* `P_mp`: mpmath, arbitrary precision (seconds), for checks of very small errors.

This notebook validates the reference itself: against the high-precision version, against
exact results, and against **Eric Agol's code** (`exoplanet-core`, Agol, Luger & Foreman-Mackey 2020)
for the quadratic law.

In [1]:
import time
import numpy as np
import mpmath as mp
import refcheck as rc

FULL = False          # True: the larger grids (minutes to hours)
t_start = time.time()

## 1a. Fast reference vs. high-precision reference

In [2]:
cases = [(0.0, 0.1), (0.5, 0.1), (0.95, 0.1), (1.05, 0.1), (0.1, 0.1), (0.81, 0.2), (1.19, 0.2),
         (0.7, 0.3), (0.995, 0.0092), (1.0091, 0.0092)]
gammas = [0, 0.25, 0.5, 1.7] if FULL else [0.3, 0.5]
worst = 0.0
for z, p in cases:
    for g in gammas:
        f = rc.P_fast(z, p, g)
        r = rc.P_mp(z, p, g, dps=20)
        worst = max(worst, float(abs(f - r) / mp.pi))      # in units of the flux of a uniform star
print(f"max |P_fast - P_mp| / pi = {worst:.1e}")
assert worst < 1e-15

max |P_fast - P_mp| / pi = 2.3e-18


## 1b. Exact cases: $\gamma=0$ (overlap area) and $\gamma=1$ inside the disk

In [3]:
def lens_area(z, p):          # overlap area of two circles, evaluated in high precision
    with mp.workdps(40):
        z, p = mp.mpf(z), mp.mpf(p)
        k0 = mp.acos((p * p + z * z - 1) / (2 * p * z))
        k1 = mp.acos((1 - p * p + z * z) / (2 * z))
        return float(p * p * k0 + k1 - mp.sqrt(4 * z * z - (1 + z * z - p * p) ** 2) / 2)

worst = 0.0
for p in [0.0092, 0.1, 0.3]:
    for z in np.linspace(0, 1 + p, 25)[1:-1]:
        exact0 = np.pi * p * p if z <= 1 - p else lens_area(z, p)
        worst = max(worst, abs(rc.P_fast(z, p, 0) - exact0) / np.pi)
        if z + p < 1:
            worst = max(worst, abs(rc.P_fast(z, p, 1) - np.pi * p * p * (1 - z * z - p * p / 2)) / np.pi)
print(f"max error against exact results: {worst:.1e}  (units of pi)")
assert worst < 1e-15

max error against exact results: 3.5e-17  (units of pi)


## 1c. Against Eric Agol's code (quadratic law)

The quadratic law is $I=(1-u_1-u_2)+(u_1+2u_2)\mu-u_2\mu^2$, i.e. exponents $\gamma=0,\tfrac12,1$.
We skip $|z-p|<10^{-12}$: at $b$ exactly one rounding step below $r$, exoplanet-core 0.4.0 returns
a flux that is off by 1 (a branch edge case; every other configuration agrees).

In [4]:
try:
    import exoplanet_core as ec
except ImportError:
    ec = None
    print("exoplanet-core not installed (pip install exoplanet-core); skipped")

if ec is not None:
    worst = 0.0
    ps = [0.0092, 0.05, 0.1, 0.2, 0.3] if FULL else [0.0092, 0.1, 0.3]
    for p in ps:
        d = np.array([1e-10, 1e-6, 1e-3])
        zs = np.unique(np.concatenate([np.linspace(0, 1 + p, 40 if FULL else 15)[:-1],
                                       1 - p - d * p, 1 - p + d * p, 1 + p - d * p]))
        zs = zs[np.abs(zs - p) > 1e-12]
        P = {g: np.array([rc.P_fast(z, p, g) for z in zs]) for g in (0, 0.5, 1)}
        for u1, u2 in [(0.4, 0.25), (1.0, 0.0), (0.0, 1.0)]:
            al, cs = [0, 1, 2], [1 - u1 - u2, u1 + 2 * u2, -u2]
            tot = 2 * np.pi * sum(c / (a + 2) for a, c in zip(al, cs))
            f_ref = 1 - sum(c * P[a / 2] for a, c in zip(al, cs)) / tot
            f_agol = ec.quad_limbdark_light_curve(u1, u2, zs, np.full(zs.size, p)) + 1
            worst = max(worst, np.max(np.abs(f_ref - f_agol)))
    print(f"max |reference - Agol| = {worst:.1e}  (flux units)")
    assert worst < 2e-15

max |reference - Agol| = 7.8e-16  (flux units)


In [5]:
print(f"notebook run time: {time.time() - t_start:.0f} s")

notebook run time: 36 s
